# 在 Bedrock AgentCore Runtime 中部署 MCP 服务器

本动手实验演示如何使用 Amazon Bedrock AgentCore Runtime 部署和使用 Model Context Protocol (MCP) 服务器，实现自定义 AI 代理工具的可扩展和安全部署。

## 概述

在本实验中，您将：
- 创建一个具有网络搜索功能的自定义 MCP 服务器
- 使用 Amazon Cognito 设置身份验证
- 将 MCP 服务器部署到 Bedrock AgentCore Runtime
- 使用 Strands Agents 测试已部署的服务器

## 前提条件

在开始本实验之前，请确保您已具备：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 MCP 服务器开发、Strands Agents 和 Bedrock AgentCore SDK 所需的包：

In [ ]:
#%pip install -q ddgs mcp strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 什么是 Bedrock AgentCore Runtime for MCP？

Amazon Bedrock AgentCore Runtime 允许您将 Model Context Protocol (MCP) 服务器部署为托管的、可扩展的服务。主要优势包括：

- **可扩展性**：根据需求自动扩展
- **安全性**：内置身份验证和授权
- **托管基础设施**：无需管理服务器或容器
- **集成能力**：与 Bedrock 服务无缝集成

MCP 服务器提供 AI 代理可用于扩展其能力的工具和资源，例如网络搜索、数据库访问或自定义业务逻辑。

### 创建自定义 MCP 服务器

让我们创建一个使用 DuckDuckGo 提供网络搜索功能的简单 MCP 服务器。该服务器将被部署到 AgentCore Runtime 以实现可扩展使用。

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from ddgs import DDGS
from ddgs.exceptions import RatelimitException, DDGSException

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

# 定义一个网络搜索工具
@mcp.tool()
def websearch(keywords: str, region: str = "us-en", max_results: int | None = None) -> list:
    """搜索网络以获取最新信息。

    Args:
        keywords (str): 搜索查询关键词。
        region (str): 搜索区域：wt-wt、us-en、uk-en、ru-ru 等。
        max_results (int | None): 返回的最大结果数。

    Returns:
        包含搜索结果的字典列表。
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
%%writefile requirements.txt
ddgs
mcp
bedrock-agentcore

### 本地测试 MCP 服务器（可选 - 如果动手实验环境中端口 8000 已被占用，请跳过）

在部署到 AgentCore Runtime 之前，先在本地测试 MCP 服务器。

### 步骤 1：启动 MCP 服务器

在终端中运行 MCP 服务器：

```bash
cd 04-bedrock-agentcore-runtime-mcp/
uv pip install -r requirements.txt
uv run mcp_server.py
```
或
```bash
cd 04-bedrock-agentcore-runtime-mcp/
pip install -r requirements.txt
python mcp_server.py
```

### 步骤 2：使用 Strands Agent 进行测试

执行以下代码来测试集成：

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# Connect to the web search MCP server
print("\n正在连接到 MCP 服务器...")
mcp_url = f"http://localhost:8000/mcp"
websearch_server = MCPClient(lambda: streamablehttp_client(mcp_url))

with websearch_server:
    mcp_tools = (websearch_server.list_tools_sync())
    print(f"Available MCP tools: {[tool.tool_name for tool in mcp_tools]}")

    # Create agent with self-built MCP tools
    agent = Agent(
        model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
        system_prompt = """你是一个生活助手，运用网络搜索的知识回答各种问题。""",
        tools=mcp_tools,
    )

    agent("什么是 Amazon Bedrock AgentCore？")

### 停止本地运行的 MCP 服务器

在本地测试 MCP 服务器后，在终端中按 `Ctrl+C` 停止本地运行的 MCP 服务器。

## 在 Bedrock AgentCore Runtime 中部署带身份验证的 MCP 服务器
现在我们将配置 MCP 服务器并将其部署到 Bedrock AgentCore Runtime。此过程包括创建依赖项、配置身份验证和部署服务。
![bedrock-agentcore-runtime-launch](images/runtime-launch.png)

### 步骤 1：设置 Amazon Cognito 入站身份验证

创建 Cognito 用户池以安全访问已部署的 MCP 服务器。

创建的组件
- **用户池**：管理用户身份
- **应用客户端**：启用应用程序身份验证
- **测试用户**：用于测试身份验证流程

In [ ]:
import boto3

region = boto3.session.Session().region_name

# Initialize Cognito client
cognito_client = boto3.client('cognito-idp', region_name=region)

# Create User Pool
user_pool_response = cognito_client.create_user_pool(
    PoolName='MCPServerPool',
    Policies={
        'PasswordPolicy': {
            'MinimumLength': 8
        }
    }
)
cognito_pool_id = user_pool_response['UserPool']['Id']

# Create App Client
app_client_response = cognito_client.create_user_pool_client(
    UserPoolId=cognito_pool_id,
    ClientName='MCPServerPoolClient',
    GenerateSecret=False,
    ExplicitAuthFlows=[
        'ALLOW_USER_PASSWORD_AUTH',
        'ALLOW_REFRESH_TOKEN_AUTH'
    ]
)
cognito_client_id = app_client_response['UserPoolClient']['ClientId']

# Create User
cognito_client.admin_create_user(
    UserPoolId=cognito_pool_id,
    Username='testuser',
    TemporaryPassword='Temp123!',
    MessageAction='SUPPRESS'
)

# Set Permanent Password
cognito_client.admin_set_user_password(
    UserPoolId=cognito_pool_id,
    Username='testuser',
    Password='MyPassword123!',
    Permanent=True
)

# Output the required values
print(f"Pool id: {cognito_pool_id}")
print(f"Discovery URL: https://cognito-idp.{region}.amazonaws.com/{cognito_pool_id}/.well-known/openid-configuration")
print(f"Client ID: {cognito_client_id}")

### 步骤 2：配置 Bedrock AgentCore Runtime

使用自动资源创建功能设置 Bedrock AgentCore Runtime 配置。

**生成的构件：**
此步骤创建必要的部署文件：
- **Dockerfile**：MCP 服务器的容器配置
- **.dockerignore**：列出 docker build 时排除的文件
- **.bedrock_agentcore.yaml**：Runtime 部署配置

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name
print(f"Using AWS region: {region}")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name="mcp_server_agentcore",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_client_id],
            "discoveryUrl": f"https://cognito-idp.{region}.amazonaws.com/{cognito_pool_id}/.well-known/openid-configuration",
        }
    }
)
print("Configuration completed ✓")

### 步骤 3：部署到 Bedrock AgentCore Runtime

使用 AWS CodeBuild 启动容器化和部署流程。

**部署流程：**
- 构建 MCP 服务器的容器化版本
- 创建所需的 AWS 资源（ECR 仓库、IAM 角色）
- 将容器镜像推送到 Amazon ECR
- 作为托管的自动扩展服务部署到 AgentCore Runtime

In [ ]:
launch_result = agentcore_runtime.launch()

### 验证部署状态

检查部署状态并等待 Runtime 就绪：

In [ ]:
import time

print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")
    
print(f"Final status: {status}")

mcp_runtime_id = launch_result.agent_id
mcp_runtime_arn = launch_result.agent_arn
ecr_repo_name = launch_result.ecr_uri.split('/')[1]
codebuild_name = launch_result.codebuild_id.split(':')[0]
print(f"MCP AgentCore Runtime ID: {mcp_runtime_id}")
print(f"MCP AgentCore Runtime ARN: {mcp_runtime_arn}")
print(f"ECR Repo for MCP AgentCore Runtime: {ecr_repo_name}")
print(f"CodeBuild Project for Strands AgentCore Runtime: {codebuild_name}")

### 测试已部署的 MCP 服务器作为 Strands Agent 的工具

现在让我们通过带有适当身份验证的 Bedrock AgentCore Runtime 端点连接到已部署的 MCP 服务器来进行测试。

首先，我们使用用户名和密码从 Cognito 身份验证获取访问令牌。

In [ ]:
import boto3

region = boto3.session.Session().region_name

# Get bearer token (access token) from Cognito Auth 
cognito_client = boto3.client('cognito-idp', region_name=boto3.session.Session().region_name)
auth_response = cognito_client.initiate_auth(
    ClientId=cognito_client_id,
    AuthFlow='USER_PASSWORD_AUTH',
    AuthParameters={
        'USERNAME': 'testuser',
        'PASSWORD': 'MyPassword123!'
    }
)
bearer_token = auth_response['AuthenticationResult']['AccessToken']
print(bearer_token)

然后，我们将访问令牌设置为请求头中的 bearer，以安全连接托管在 AgentCore Runtime 中的 MCP 服务器。

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

region = boto3.session.Session().region_name

mcp_runtime_arn = launch_result.agent_arn
encoded_arn = mcp_runtime_arn.replace(':', '%3A').replace('/', '%2F')

# Connect to the Web Search MCP server
print("\n正在连接到 MCP 服务器...")
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
headers = {
    "Authorization": f"Bearer {bearer_token}",
    #"Content-Type": "application/json"
}
websearch_server = MCPClient(lambda: streamablehttp_client(mcp_url, headers))

with websearch_server:
    mcp_tools = (websearch_server.list_tools_sync())
    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")

    # Create agent with self-built MCP tools
    agent = Agent(
        model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
        system_prompt = """你是一个生活助手，运用网络搜索的知识回答各种问题。""",
        tools=mcp_tools,
    )

    agent("什么是 Amazon Bedrock AgentCore？")

让我们查看代理循环的详细执行流程，以了解代理如何处理请求并生成响应：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## 资源清理（可选）

清理已部署的资源：

In [ ]:
import boto3
import os

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr',region_name=region)
codebuild_client = boto3.client('codebuild',region_name=region)
cognito_client = boto3.client('cognito-idp', region_name=region)

try:
    print("Deleting AgentCore Runtime...")
    agentcore_control_client.delete_agent_runtime(agentRuntimeId=mcp_runtime_id)
    print("✓ AgentCore Runtime deletion initiated")

    print("Deleting ECR repository...")
    ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
    print("✓ ECR repository deleted")

    print("Deleting CodeBuild Project...")
    codebuild_client.delete_project(name=codebuild_name)
    print("✓ CodeBuild Project deleted")

    print("Deleting Cognito User Pool...")
    cognito_client.delete_user_pool(UserPoolId=cognito_pool_id)
    print("✓ Cognito User Pool deleted")

    print("Deleting Bedrock AgentCore configuration file...")
    os.remove(".bedrock_agentcore.yaml") 
    print("✓ .bedrock_agentcore.yaml deleted")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了：

- ✅ 使用 DuckDuckGo 创建了具有网络搜索功能的自定义 MCP 服务器
- ✅ 设置了 Amazon Cognito 以实现 MCP 服务器的安全身份验证
- ✅ 配置并将 MCP 服务器部署到 Bedrock AgentCore Runtime
- ✅ 将已部署的 MCP 服务器与 Strands Agents 集成，实现 AI 驱动的工作流
  
## AgentCore Runtime for MCP 的主要优势

- **可扩展部署**：无需服务器管理的 MCP 服务器托管基础设施
- **安全身份验证**：内置支持 Cognito 和其他身份验证方法
- **轻松集成**：与 Strands Agents 和其他 AI 框架无缝连接
- **生产就绪**：企业级可靠性和监控能力